# Phase 1: Data Analysis & Understanding
## Nemotron Reasoning Challenge

In [ ]:
import pandas as pd
import re
from collections import Counter

## 1. Load and Explore Training Data

In [ ]:
train_df = pd.read_csv('data/train.csv')
print(f"Training samples: {len(train_df)}")
print(f"Columns: {train_df.columns.tolist()}")
train_df.head()

In [ ]:
print("\n=== Sample entries ===")
for i in range(3):
    print(f"\n--- Puzzle {i+1} ---")
    print(f"ID: {train_df.iloc[i]['id']}")
    print(f"Prompt: {train_df.iloc[i]['prompt'][:500]}...")
    print(f"Answer: {train_df.iloc[i]['answer']}")

## 2. Classify Puzzle Types

In [ ]:
def classify_puzzle(prompt):
    """Classify puzzle type based on prompt keywords."""
    prompt_lower = prompt.lower()
    
    if 'bit' in prompt_lower and ('binary' in prompt_lower or 'bit' in prompt_lower):
        return 'bit_manipulation'
    elif 'encrypt' in prompt_lower or 'decrypt' in prompt_lower:
        if any(c in prompt_lower for c in ['->', 'queen', 'dragon', 'castle', 'wizard']):
            return 'text_encryption'
    elif 'numeral' in prompt_lower or ('roman' in prompt_lower) or any(x in prompt for x in ['XI', 'XV', 'XCIV', 'XIX']):
        return 'numeral_conversion'
    elif 'unit' in prompt_lower or 'conversion' in prompt_lower or 'm' in prompt_lower:
        return 'unit_conversion'
    elif 'equation' in prompt_lower or 'algebra' in prompt_lower:
        return 'algebra'
    else:
        return 'unknown'

In [ ]:
train_df['puzzle_type'] = train_df['prompt'].apply(classify_puzzle)
print("Puzzle type distribution:")
print(train_df['puzzle_type'].value_counts())

## 3. Analyze Specific Puzzle Types

In [ ]:
# Look at bit manipulation examples
bit_puzzles = train_df[train_df['puzzle_type'] == 'bit_manipulation']
print(f"Bit manipulation puzzles: {len(bit_puzzles)}")
print("\n=== Sample Bit Manipulation Puzzles ===")
for i in range(3):
    print(f"\n--- Example {i+1} ---")
    print(bit_puzzles.iloc[i]['prompt'][:800])
    print(f"Answer: {bit_puzzles.iloc[i]['answer']}")

In [ ]:
# Look at text encryption examples
text_puzzles = train_df[train_df['puzzle_type'] == 'text_encryption']
print(f"Text encryption puzzles: {len(text_puzzles)}")
print("\n=== Sample Text Encryption Puzzles ===")
for i in range(2):
    print(f"\n--- Example {i+1} ---")
    print(text_puzzles.iloc[i]['prompt'][:700])
    print(f"Answer: {text_puzzles.iloc[i]['answer']}")

In [ ]:
# Look at numeral conversion examples
numeral_puzzles = train_df[train_df['puzzle_type'] == 'numeral_conversion']
print(f"Numeral conversion puzzles: {len(numeral_puzzles)}")
print("\n=== Sample Numeral Conversion Puzzles ===")
for i in range(3):
    print(f"\n--- Example {i+1} ---")
    print(numeral_puzzles.iloc[i]['prompt'][:600])
    print(f"Answer: {numeral_puzzles.iloc[i]['answer']}")

## 4. Deeper Analysis: Bit Manipulation Rules

In [ ]:
def extract_io_examples(prompt):
    """Extract input->output examples from prompt."""
    # Match patterns like "01010001 -> 11011101"
    pattern = r'([01]{8})\s*->\s*([01]{8})'
    matches = re.findall(pattern, prompt)
    return matches

In [ ]:
# Extract all IO examples from bit manipulation puzzles
bit_examples = []
for _, row in bit_puzzles.iterrows():
    examples = extract_io_examples(row['prompt'])
    bit_examples.extend(examples)
    
print(f"Total bit manipulation examples: {len(bit_examples)}")
print("\n=== First 10 Examples ===")
for inp, out in bit_examples[:10]:
    print(f"{inp} -> {out}")

In [ ]:
# Try to infer the bit manipulation rule
# Common operations: reverse, complement, rotate, swap nibbles

def analyze_bit_rule(examples):
    """Try to find a simple bit operation that maps input to output."""
    results = []
    
    for inp, out in examples[:20]:
        inp_int = int(inp, 2)
        out_int = int(out, 2)
        
        # Try various operations
        ops = {
            'reverse': int(inp[::-1], 2),
            'complement': (~inp_int) & 0xFF,
            'swap_nibbles': ((inp_int & 0xF0) >> 4) | ((inp_int & 0x0F) << 4),
            'rotate_left': ((inp_int << 1) | (inp_int >> 7)) & 0xFF,
            'rotate_right': ((inp_int >> 1) | (inp_int << 7)) & 0xFF,
        }
        
        matches = [k for k, v in ops.items() if v == out_int]
        results.append((inp, out, matches))
    
    return results

In [ ]:
results = analyze_bit_rule(bit_examples)
print("=== Possible Operations per Example ===")
for inp, out, matches in results:
    print(f"{inp} -> {out} : {matches if matches else 'unknown'}")

## 5. Analyze Text Encryption

In [ ]:
# Extract text encryption mapping
def extract_text_examples(prompt):
    """Extract encrypted->decrypted examples."""
    # Match patterns like "ucoov pwgtfyoqg vorq yrjjoe -> queen discovers near valley"
    lines = prompt.split('\n')
    examples = []
    for line in lines:
        if '->' in line and 'Now' not in line:
            parts = line.split('->')
            if len(parts) == 2:
                encrypted = parts[0].strip()
                decrypted = parts[1].strip()
                examples.append((encrypted, decrypted))
    return examples

In [ ]:
text_examples = []
for _, row in text_puzzles.head(10).iterrows():
    examples = extract_text_examples(row['prompt'])
    print(f"\nPuzzle: {row['id']}")
    for enc, dec in examples:
        print(f"  {enc} -> {dec}")
        text_examples.append((enc, dec))

## 6. Analyze Numeral Conversion

In [ ]:
# Try to understand the numeral system
def extract_number_examples(prompt):
    """Extract number -> numeral examples."""
    lines = prompt.split('\n')
    examples = []
    for line in lines:
        # Match patterns like "11 -> XI"
        match = re.search(r'(\d+)\s*->\s*([A-Z]+)', line)
        if match:
            examples.append((match.group(1), match.group(2)))
    return examples

In [ ]:
num_examples = []
for _, row in numeral_puzzles.head(5).iterrows():
    examples = extract_number_examples(row['prompt'])
    print(f"\nPuzzle: {row['id']}")
    print(f"Prompt: {row['prompt'][:400]}")
    print(f"Examples: {examples}")
    print(f"Answer: {row['answer']}")
    num_examples.extend(examples)

## 7. Test Data Overview

In [ ]:
test_df = pd.read_csv('data/test.csv')
print(f"Test samples: {len(test_df)}")
test_df['puzzle_type'] = test_df['prompt'].apply(classify_puzzle)
print("\nTest puzzle type distribution:")
print(test_df['puzzle_type'].value_counts())

## Summary
- Training data contains ~69K puzzles
- Main puzzle types: bit manipulation, text encryption, numeral conversion, unit conversion, algebra
- Need to analyze patterns more thoroughly to understand transformation rules
- Next: Test baseline model with prompting approaches